# SAE Feature Analysis — Activation Distributions by BSP Class

The heavy computation (encoding + feature×BSP matching) is done **once** by `sae_eval.py evaluate` and cached to disk alongside the SAE checkpoint:

| File | Contents |
|------|----------|
| `saes/{game}/{run_id}_h.pt` | `(N × d_dict)` activation matrix |
| `saes/{game}/{run_id}_matching-{animal}.pt` | Full `(d_dict × num_bsps)` precision / recall / F1 tensors |

This notebook **only loads those files** — switching models means changing `CHECKPOINT` in the config cell and re-running. No re-encoding.

**Pipeline:**
1. Load BSP schema (feature names)
2. Load cached `h` and `matching`
3. Select top-k BSPs by best F1
4. Plot activation histograms split by BSP class, with threshold τ annotated
5. Threshold analysis table (TPR / FPR / P / R / F1 at τ)


## 1 — Imports

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch

# Make project root importable from notebooks/
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

print("Ready. ROOT =", ROOT)

## 2 — Configuration

Change `CHECKPOINT` to any valid SAE `.pt` file. Everything else auto-resolves from checkpoint metadata.

In [ ]:
# ── Checkpoint to analyse ─────────────────────────────────────────────────
CHECKPOINT = ROOT / "saes/quarto/arnold_beta-jumprelu-t64-exp8-fc1.pt"

# ── BSP set ──────────────────────────────────────────────────────────────
BSP_ANIMAL = "gorilla"  # "gorilla" (164) or "fox" (87)

# ── Visualisation ────────────────────────────────────────────────────────
TOP_K_BSPS = 12  # how many top BSPs to plot
THRESHOLD = 0.0  # firing threshold τ  (h > τ  → active)
BINS = 60  # histogram bins

# ── Output ───────────────────────────────────────────────────────────────
OUT_DIR = ROOT / "notebooks" / "feature_analysis_outputs"
OUT_DIR.mkdir(exist_ok=True)

stem = CHECKPOINT.stem  # e.g. "arnold_beta-jumprelu-t64-exp8-fc1"
game = stem.split("-")[0].replace("_beta", "").replace("arnold", "quarto")  # fallback
saes_dir = CHECKPOINT.parent  # saes/quarto/
cache_dir = saes_dir / "cache"  # saes/quarto/cache/

# Override game properly by reading checkpoint metadata
import torch as _torch

_meta = _torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
if isinstance(_meta, dict) and "metadata" in _meta:
    game = _meta["metadata"].get("game", game)
    hook = _meta["metadata"].get("hook", "fc1")
elif isinstance(_meta, dict) and "game" in _meta:
    game = _meta.get("game", game)
    hook = _meta.get("hook", "fc1")
del _meta, _torch

print(f"Checkpoint : {CHECKPOINT.name}")
print(f"Game       : {game}  |  Hook: {hook}")
print(f"BSP set    : {BSP_ANIMAL}")
print(f"Caches in  : {cache_dir.relative_to(ROOT)}")

## 3 — Load BSP Schema

Only the BSP schema (feature names) is needed here. `h` and the feature×BSP matching are loaded from the caches written by `sae_eval.py evaluate`.

In [ ]:
data_dir = ROOT / f"data/{game}"

# BSP schema — for human-readable feature names only
schema_files = sorted(data_dir.glob(f"bsp_schema-{BSP_ANIMAL}_*.json"))
label_files = sorted(data_dir.glob(f"bsp_labels-{BSP_ANIMAL}_*.pt"))
assert schema_files, f"No BSP schema found for animal '{BSP_ANIMAL}' in {data_dir}"
assert label_files, f"No BSP label file found for animal '{BSP_ANIMAL}' in {data_dir}"

bsp_schema = json.loads(schema_files[0].read_text())
bsp_labels = torch.load(label_files[0], map_location="cpu").float()  # (N, num_bsps)
bsp_names = [b["name"] for b in bsp_schema]
num_bsps = len(bsp_names)

print(f"BSP set : {schema_files[0].name}  |  {num_bsps} BSPs")
print(f"Labels  : {bsp_labels.shape}")

## 4 — Load Cached h

`sae_eval.py evaluate` saves the full `(N × d_dict)` activation matrix to `saes/{game}/{run_id}_h.pt`. We just load it — no re-encoding.

In [ ]:
h_cache = cache_dir / f"{stem}_h.pt"
assert h_cache.exists(), (
    f"Cache not found: {h_cache}\n"
    f"Run first:  python sae_eval.py evaluate {CHECKPOINT.relative_to(ROOT)}"
)

h = torch.load(h_cache, map_location="cpu", weights_only=True)  # (N, d_dict)
print(f"h shape : {tuple(h.shape)}")
print(
    f"Mean active features per sample : {(h > THRESHOLD).float().sum(dim=1).mean().item():.1f}"
)

## 5 — Load Cached Feature × BSP Matching

`sae_eval.py evaluate` also saves `saes/{game}/{run_id}_matching-{animal}.pt` with the full `(d_dict × num_bsps)` precision / recall / F1 matrices.

In [ ]:
from lib.sae.eval import FeatureBSPMatching

matching_cache = cache_dir / f"{stem}_matching-{BSP_ANIMAL}.pt"
assert matching_cache.exists(), (
    f"Cache not found: {matching_cache}\n"
    f"Run first:  python sae_eval.py evaluate {CHECKPOINT.relative_to(ROOT)}"
)

c = torch.load(matching_cache, map_location="cpu", weights_only=False)
matching = FeatureBSPMatching(
    precision=c["precision"],  # (d_dict, num_bsps)
    recall=c["recall"],  # (d_dict, num_bsps)
    f1=c["f1"],  # (d_dict, num_bsps)
    best_f1_per_bsp=c["best_f1_per_bsp"],  # (num_bsps,)
    best_feature_per_bsp=c["best_feature_per_bsp"],  # (num_bsps,)
)
print(f"Loaded matching: {matching.f1.shape[0]} features × {matching.f1.shape[1]} BSPs")

# Summary CSV (for quick inspection outside the notebook)
df_summary = (
    pd.DataFrame(
        {
            "bsp_name": bsp_names,
            "best_f1": matching.best_f1_per_bsp.numpy(),
            "best_feature": matching.best_feature_per_bsp.numpy().astype(int),
            "best_precision": matching.precision[
                matching.best_feature_per_bsp, torch.arange(num_bsps)
            ].numpy(),
            "best_recall": matching.recall[
                matching.best_feature_per_bsp, torch.arange(num_bsps)
            ].numpy(),
        }
    )
    .sort_values("best_f1", ascending=False)
    .reset_index(drop=True)
)

csv_path = saes_dir / f"{stem}_matching-{BSP_ANIMAL}.csv"
df_summary.to_csv(csv_path, index=False)
print(f"Summary CSV → {csv_path.relative_to(ROOT)}")
df_summary.head(20)

## 6 — Select Top-k BSPs

In [ ]:
# Rank BSPs by best F1 and take the top K
top_k = min(TOP_K_BSPS, num_bsps)
top_bsp_indices = matching.best_f1_per_bsp.argsort(descending=True)[:top_k].tolist()

print(f"Top {top_k} BSPs by best SAE feature F1:\n")
print(
    f"{'#':>3}  {'BSP':40}  {'Best F1':>8}  {'Precision':>10}  {'Recall':>8}  {'Feature':>8}"
)
print("─" * 80)
for rank, j in enumerate(top_bsp_indices):
    feat = int(matching.best_feature_per_bsp[j].item())
    f1 = matching.best_f1_per_bsp[j].item()
    prec = matching.precision[feat, j].item()
    rec = matching.recall[feat, j].item()
    print(
        f"{rank+1:>3}  {bsp_names[j]:40}  {f1:>8.4f}  {prec:>10.4f}  {rec:>8.4f}  {feat:>8d}"
    )

## 7 — Activation Histograms by BSP Class

For each top BSP **j**:
- Find the best matching SAE feature index `feat = best_feature_per_bsp[j]`
- Split the `N` samples into **positive** (BSP active) and **negative** (BSP inactive)
- Plot overlapping histograms of `h[:, feat]` for both groups
- Draw the firing threshold `τ` as a vertical line

In [ ]:
COLS = 3
rows = (top_k + COLS - 1) // COLS
fig, axes = plt.subplots(rows, COLS, figsize=(6 * COLS, 4 * rows))
axes = axes.flatten()

h_np = h.numpy()  # (N, d_dict)
labels_np = bsp_labels.numpy()  # (N, num_bsps)

for ax_idx, j in enumerate(top_bsp_indices):
    ax = axes[ax_idx]
    feat = int(matching.best_feature_per_bsp[j].item())

    act = h_np[:, feat]  # activation values for this feature
    pos_mask = labels_np[:, j] > 0.5  # BSP-positive samples
    neg_mask = ~pos_mask  # BSP-negative samples

    act_pos = act[pos_mask]
    act_neg = act[neg_mask]

    # Determine common bin range
    all_act = act[act > 0] if (act > 0).any() else act
    x_max = float(np.percentile(all_act, 99)) if len(all_act) > 0 else 1.0
    bins = np.linspace(0, x_max, BINS + 1)

    ax.hist(
        act_neg,
        bins=bins,
        alpha=0.55,
        color="#5B9BD5",
        label=f"BSP=0  (n={neg_mask.sum():,})",
        density=True,
    )
    ax.hist(
        act_pos,
        bins=bins,
        alpha=0.70,
        color="#ED7D31",
        label=f"BSP=1  (n={pos_mask.sum():,})",
        density=True,
    )

    # Threshold line
    ax.axvline(THRESHOLD, color="crimson", lw=1.5, ls="--", label=f"τ = {THRESHOLD}")

    f1 = matching.best_f1_per_bsp[j].item()
    prec = matching.precision[feat, j].item()
    rec = matching.recall[feat, j].item()

    ax.set_title(
        f"{bsp_names[j]}\nFeat #{feat} · F1={f1:.3f} · P={prec:.3f} · R={rec:.3f}",
        fontsize=9,
    )
    ax.set_xlabel("Activation value", fontsize=8)
    ax.set_ylabel("Density", fontsize=8)
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=7)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))

# Hide unused subplots
for ax_idx in range(top_k, len(axes)):
    axes[ax_idx].set_visible(False)

fig.suptitle(
    f"{stem}\nTop {top_k} BSPs — feature activation distribution by class",
    fontsize=11,
    y=1.01,
)
fig.tight_layout()

plot_path = OUT_DIR / f"histograms-{stem}-{BSP_ANIMAL}-top{top_k}.png"
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Saved → {plot_path.relative_to(ROOT)}")
plt.show()

## 8 — Threshold Analysis: Active / Inactive Ratio per Class

For each top BSP, apply the threshold $\tau$ to binarize the feature activation and report:
- **True Positive Rate** (BSP=1, feature fires)  
- **False Positive Rate** (BSP=0, feature fires)  
- **Precision / Recall / F1** at threshold $\tau$

This shows whether the threshold $\tau = 0$ is the right operating point or whether a higher threshold would improve selectivity.

In [ ]:
records = []
for j in top_bsp_indices:
    feat = int(matching.best_feature_per_bsp[j].item())
    act = h_np[:, feat]
    y = labels_np[:, j]

    fired = (act > THRESHOLD).astype(float)

    n_pos = y.sum()
    n_neg = len(y) - n_pos

    tpr = fired[y > 0.5].mean() if n_pos > 0 else 0.0  # recall
    fpr = fired[y < 0.5].mean() if n_neg > 0 else 0.0  # false positive rate

    tp = ((fired == 1) & (y == 1)).sum()
    fp = ((fired == 1) & (y == 0)).sum()
    fn = ((fired == 0) & (y == 1)).sum()
    eps = 1e-8
    prec_thresh = tp / (tp + fp + eps)
    rec_thresh = tp / (tp + fn + eps)
    f1_thresh = 2 * prec_thresh * rec_thresh / (prec_thresh + rec_thresh + eps)

    records.append(
        {
            "BSP": bsp_names[j],
            "Feature": feat,
            "Best F1 (match)": round(float(matching.best_f1_per_bsp[j].item()), 4),
            "TPR (recall)": round(float(tpr), 4),
            "FPR": round(float(fpr), 4),
            "Precision@τ": round(float(prec_thresh), 4),
            "Recall@τ": round(float(rec_thresh), 4),
            "F1@τ": round(float(f1_thresh), 4),
            "n_pos": int(n_pos),
            "n_neg": int(n_neg),
        }
    )

df_thresh = pd.DataFrame(records)
df_thresh.to_csv(
    OUT_DIR / f"threshold_analysis-{stem}-{BSP_ANIMAL}-top{top_k}.csv", index=False
)
df_thresh

### Optional: Precision–Recall curve as τ varies

Sweep $\tau$ from 0 to the 99th percentile of the feature's activation to see the best operating point for each top BSP.